# LangChain: Chains (Groq Llama 3.1)

## Outline
* **LLMChain** - Basic chain combining LLM + Prompt
* **Sequential Chains** - Running chains one after another
  * SimpleSequentialChain - Single input/output per chain
  * SequentialChain - Multiple inputs/outputs per chain
* **Router Chain** - Route inputs to specialized chains

**Model Used:** Groq Llama-3.1-8b-instant

Chains are the fundamental building blocks in LangChain that combine LLMs with prompts to create complex workflows.


In [ ]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv pandas -q

In [ ]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


In [ ]:
# Cell 4: Load Sample Data

# Load product reviews data
# Make sure Data.csv is in the same directory or provide the full path
df = pd.read_csv('Data.csv')

print("Dataset loaded successfully!")
print(f"Number of reviews: {len(df)}")
print("\nFirst few rows:")
df.head()


In [ ]:
# Cell 5: Initialize Groq LLM

from langchain_groq import ChatGroq

# Set model
llm_model = "llama-3.1-8b-instant"

print(f"✅ Model: {llm_model}")


## LLMChain

The **LLMChain** is the most basic chain in LangChain. It combines:
- A **prompt template** (with variables)
- An **LLM** (language model)

Together they create a reusable chain that formats prompts and gets responses.


In [ ]:
# Cell 7: Create Basic LLMChain

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

# Initialize LLM with high temperature for creative outputs
llm = ChatGroq(temperature=0.9, model=llm_model, groq_api_key=groq_api_key)

# Create prompt template
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# Create the chain by combining LLM + Prompt
chain = LLMChain(llm=llm, prompt=prompt)

print("✅ LLMChain created successfully")


In [ ]:
# Cell 8: Run LLMChain with Sample Product

# Define a product
product = "Queen Size Sheet Set"

# Run the chain
response = chain.run(product)

print(f"Product: {product}")
print(f"Suggested Company Name: {response}")


## SimpleSequentialChain

This chain runs multiple chains in sequence where:
- Each chain has **one input** and **one output**
- The output of one chain becomes the input of the next

**Use Case:** When you need to process data through multiple steps in a pipeline.


In [ ]:
# Cell 10: Create SimpleSequentialChain

from langchain.chains import SimpleSequentialChain

# Initialize LLM
llm = ChatGroq(temperature=0.9, model=llm_model, groq_api_key=groq_api_key)

# Chain 1: Generate company name from product
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
chain_one = LLMChain(llm=llm, prompt=first_prompt)

# Chain 2: Generate company description from company name
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company: {company_name}"
)
chain_two = LLMChain(llm=llm, prompt=second_prompt)

# Combine into SimpleSequentialChain
overall_simple_chain = SimpleSequentialChain(
    chains=[chain_one, chain_two],
    verbose=True
)

print("✅ SimpleSequentialChain created with 2 chains")


In [ ]:
# Cell 11: Run SimpleSequentialChain

# Run the chain with our product
result = overall_simple_chain.run(product)

print("\nFinal Output:")
print(result)


## SequentialChain

Unlike SimpleSequentialChain, this advanced chain supports:
- **Multiple inputs** per chain
- **Multiple outputs** per chain
- **Named variables** to track intermediate results

**Use Case:** Complex workflows where chains need access to multiple previous outputs.


In [ ]:
# Cell 13: Create SequentialChain - Chain 1 (Translation)

from langchain.chains import SequentialChain

# Initialize LLM
llm = ChatGroq(temperature=0.9, model=llm_model, groq_api_key=groq_api_key)

# Chain 1: Translate review to English
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:\n\n{Review}"
)
chain_one = LLMChain(
    llm=llm, 
    prompt=first_prompt, 
    output_key="English_Review"  # Named output
)

print("✅ Chain 1: Translation chain created")


In [ ]:
# Cell 14: Create SequentialChain - Chain 2 (Summarization)

# Chain 2: Summarize the English review
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
chain_two = LLMChain(
    llm=llm, 
    prompt=second_prompt, 
    output_key="summary"  # Named output
)

print("✅ Chain 2: Summarization chain created")


In [ ]:
# Cell 15: Create SequentialChain - Chain 3 (Language Detection)

# Chain 3: Detect original language
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
chain_three = LLMChain(
    llm=llm, 
    prompt=third_prompt,
    output_key="language"  # Named output
)

print("✅ Chain 3: Language detection chain created")


In [ ]:
# Cell 16: Create SequentialChain - Chain 4 (Follow-up)

# Chain 4: Generate follow-up response (uses 2 inputs!)
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
chain_four = LLMChain(
    llm=llm, 
    prompt=fourth_prompt,
    output_key="followup_message"  # Named output
)

print("✅ Chain 4: Follow-up response chain created")


In [ ]:
# Cell 17: Combine into SequentialChain

# Combine all 4 chains into one sequential chain
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],  # Initial input
    output_variables=["English_Review", "summary", "followup_message"],  # Outputs to return
    verbose=True
)

print("✅ SequentialChain created with 4 sub-chains")
print("\nChain Flow:")
print("1. Review → English_Review")
print("2. English_Review → summary")
print("3. Review → language")
print("4. summary + language → followup_message")


In [ ]:
# Cell 18: Run SequentialChain on Real Review

# Get a review from the dataset (index 5)
review = df.Review[5]

print("Original Review:")
print("="*60)
print(review)
print("\n" + "="*60 + "\n")

# Run the sequential chain
result = overall_chain(review)

print("\nFinal Results:")
print("="*60)
for key, value in result.items():
    if key != 'Review':  # Skip echoing the input
        print(f"\n{key}:")
        print(value)


## Router Chain

A **Router Chain** intelligently routes inputs to specialized sub-chains based on the input content.

**How it works:**
1. Input comes in
2. Router LLM decides which expert chain to use
3. Input is sent to that specialized chain
4. Response is returned

**Use Case:** Building a multi-domain assistant (e.g., one expert for physics, another for math, etc.)


In [ ]:
# Cell 20: Define Subject-Specific Prompt Templates

# Physics expert prompt
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise \
and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""

# Math expert prompt
math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, \
answer the component parts, and then put them together \
to answer the broader question.

Here is a question:
{input}"""

# History expert prompt
history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people, \
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence \
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""

# Computer Science expert prompt
computerscience_template = """You are a successful computer scientist. \
You have a passion for creativity, collaboration, \
forward-thinking, confidence, strong problem-solving capabilities, \
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

print("✅ Expert prompt templates defined for 4 subjects")


In [ ]:
# Cell 21: Create Prompt Info Dictionary

# Define metadata for each expert chain
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "history", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

print("✅ Prompt info configured:")
for info in prompt_infos:
    print(f"  - {info['name']}: {info['description']}")


In [ ]:
# Cell 22: Import Router Chain Components

from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate

print("✅ Router chain components imported")


In [ ]:
# Cell 23: Create Destination Chains

# Initialize LLM with temperature=0 for consistent routing
llm = ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)

# Create a chain for each expert domain
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

print(f"✅ Created {len(destination_chains)} destination chains:")
for name in destination_chains.keys():
    print(f"  - {name}")


In [ ]:
# Cell 24: Create Default Chain

# Default chain for questions that don't match any expert
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

print("✅ Default chain created (for non-specialized questions)")


In [ ]:
# Cell 25: Define Router Template

# Create destinations string for the router
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

# Router template - instructs the LLM how to route
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising \
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
REMEMBER: "destination" MUST be one of the candidate prompt names below OR "DEFAULT".
REMEMBER: "next_inputs" can just be the original input if no modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (must include ```json) >>"""

Format the template with our destinations
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
destinations=destinations_str
)

print("✅ Router template created")
print("\nAvailable destinations:")
print(destinations_str)

In [ ]:
# Cell 26: Create Router Chain

# Create router prompt
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

# Create the router chain
router_chain = LLMRouterChain.from_llm(llm, router_prompt)

print("✅ Router chain created")

In [ ]:
# Cell 27: Create MultiPromptChain

# Combine router + destination chains + default chain
chain = MultiPromptChain(
    router_chain=router_chain, 
    destination_chains=destination_chains, 
    default_chain=default_chain, 
    verbose=True
)

print("✅ MultiPromptChain created successfully")
print("\nThis chain will:")
print("1. Analyze the input question")
print("2. Route to the appropriate expert (physics/math/history/cs)")
print("3. Return the expert's response")


In [ ]:
# Cell 28: Test Router Chain - Physics Question

# Ask a physics question
response = chain.run("What is black body radiation?")

print("\nResponse:")
print(response)


In [ ]:
# Cell 29: Test Router Chain - Math Question

# Ask a math question
response = chain.run("What is 2 + 2?")

print("\nResponse:")
print(response)


In [ ]:
# Cell 30: Test Router Chain - Non-Expert Question

# Ask a question outside the expert domains (e.g., biology)
response = chain.run("Why does DNA replicate?")

print("\nResponse:")
print(response)
print("\n⚠️ This should route to DEFAULT chain since biology isn't in our expert domains")


In [ ]:
# Cell 31: Test Router Chain - History Question

# Ask a history question
response = chain.run("Who was the first president of the United States?")

print("\nResponse:")
print(response)


In [ ]:
# Cell 32: Test Router Chain - Computer Science Question

# Ask a computer science question
response = chain.run("What is the time complexity of binary search?")

print("\nResponse:")
print(response)


## Summary: Chain Types in LangChain

### 1. **LLMChain**
- **Simplest chain**: LLM + Prompt Template
- Use when you need a single-step prompt with variables

### 2. **SimpleSequentialChain**
- **Pipeline of chains**: Output of chain N → Input of chain N+1
- Limitation: Only 1 input and 1 output per chain
- Use for linear workflows (e.g., translate → summarize)

### 3. **SequentialChain**
- **Advanced pipeline**: Multiple inputs/outputs per chain
- Named variables track intermediate results
- Use for complex workflows where chains need multiple previous outputs

### 4. **Router Chain**
- **Intelligent routing**: Directs questions to specialized expert chains
- Uses LLM to decide which sub-chain to invoke
- Use for multi-domain assistants or specialized knowledge bases

---

## Key Concepts

| Concept | Description |
|---------|-------------|
| **Chain** | Combines LLM + Prompt + Logic into reusable component |
| **Sequential Flow** | Chains execute one after another |
| **Named Variables** | Track intermediate outputs with `output_key` |
| **Routing** | LLM decides which expert chain to use |
| **Default Chain** | Fallback when no expert matches |

---